# Modelagem — Score de detração pós-viagem

Este notebook treina e avalia o modelo que estima, para cada jornada encerrada, a probabilidade de o Cliente se tornar Detrator. Ele começa onde o `pre-processamento.ipynb` termina: consome a base analítica já integrada e não repete nenhuma etapa de limpeza ou integração.

Princípios adotados:

- consumir a base analítica pronta, sem reprocessar as fontes brutas;
- importar a lógica de seleção, divisão e transformação de `scripts/preprocessamento_nps.py`, nunca reescrevê-la aqui;
- ajustar imputação, codificação e escala somente no treino;
- manter o conjunto de teste intocado até a etapa de avaliação;
- versionar o notebook sem saídas de execução e sem nenhum dado do parceiro.

O notebook é preenchido por etapas, cada uma com card próprio no board:

| Seção | Conteúdo | Card |
|---|---|---|
| 1 | Contrato de entrada e carga da matriz | #100 |
| 2 | Baselines de referência | #101 |
| 3 | Validação cruzada por Cliente | #102 |
| 4 | Primeiro modelo candidato | #103 |
| 5 | Ajuste de hiperparâmetros | #104 |
| 6 | Métricas de ordenação | #105 |
| 7 | Limiar operacional e matriz de confusão | #106 |
| 8 | Curvas ROC e precisão contra cobertura | #107 |
| 9 | Tabela comparativa | #108 |

## 1. Contrato de entrada

Esta seção fixa o que a etapa de carga e organização dos dados precisa entregar para que a modelagem rode. O contrato existe para que as duas frentes avancem em paralelo: enquanto a organização não estiver pronta, basta que o arquivo abaixo exista com estas colunas para que todo o restante do notebook funcione sem alteração.

**Arquivo esperado:** `data/processed/base_analitica.parquet`, produzido pela seção 13 do `notebooks/pre-processamento.ipynb`. No Colab, o mesmo arquivo pode vir do Drive; a busca de caminho está na célula seguinte e não exige edição de código.

**Colunas obrigatórias.** A base precisa carregar três grupos:

| Grupo | Colunas | Para que serve |
|---|---|---|
| Alvo | `DETRATOR` | variável a ser predita, binária |
| Chaves de partição | `DATA_STD_CONVERTIDA`, `ID_GOLDENRECORD` | corte temporal e agrupamento por Cliente |
| Features | as 11 de `FEATURES_SCORE_POS_VIAGEM` | matriz do modelo |

As 11 features são a allowlist do score pós-viagem definida no módulo de pré-processamento e não são reescritas aqui: `PERFIL_TUDOAZUL`, `VOO_TIPO`, `TIPO_ENTRETENIMENTO`, `CANAL_COMPRA`, `SEGMENTO`, `ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA`, `CANCELAMENTO_VOO`, `ANTECEDENCIA_CANCELAMENTO`, `TEMPO_VOO`, `QTDE_VIAGENS_12M`. Qualquer mudança nessa lista é decisão de modelagem e acontece no módulo, não no notebook.

**Índices das partições.** A divisão não é aleatória. `dividir_treino_teste_temporal_por_cliente` reserva os meses finais para teste e retira do treino todo Cliente que aparece no teste, porque a base tem dependência intracliente e um Cliente presente nos dois lados inflaria o desempenho estimado. A função devolve os índices do pandas, e é por eles que treino e teste são referenciados no restante do notebook, nunca por posição.

**O que não pode entrar na matriz do modelo.** Nenhuma coluna iniciada por `NPS_` ou `SUB_`. São respostas coletadas no mesmo instrumento que origina o alvo, e o contrato temporal da seção 4.2.3 da documentação as exclui do modelo.

A proibição vale para a matriz, e não para a base analítica. A base carrega essas colunas por construção: `DETRATOR` é derivado de `NPS_PRINCIPAL` dentro da própria `preparar_base_analitica`, e a persistência em Parquet da seção 13 do pré-processamento não descarta coluna alguma. Exigir uma base sem elas faria o notebook falhar sobre uma entrada correta. Por isso as duas verificações têm papéis distintos, e isso é deliberado:

| Célula | Papel | Interrompe a execução |
|---|---|---|
| 1.3 | diagnóstico da base analítica: confere colunas obrigatórias e features da allowlist, e apenas relata quantas colunas de pesquisa vieram junto | sim para coluna obrigatória ausente, não para coluna de pesquisa presente |
| 1.4 | contrato da matriz do modelo: confere que o conjunto de features é exatamente a allowlist e que nenhuma coluna de pesquisa entrou, na entrada e na saída do pré-processador | sim |

### 1.1. Dependências e módulo de pré-processamento

In [ ]:
!pip install -q pandas numpy scikit-learn pyarrow matplotlib

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Inclui a pasta do módulo tanto no projeto local quanto no Colab. O break evita
# que duas cópias de preprocessamento_nps.py fiquem no caminho ao mesmo tempo,
# o que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import (
    FEATURES_SCORE_POS_VIAGEM,
    criar_preprocessador_modelagem,
    validar_parquet,
)

SEMENTE = 42
np.random.seed(SEMENTE)

# O contrato da seção 1 ancora em 11 features. Uma mudança silenciosa na
# allowlist do módulo precisa falhar aqui, e não apenas aparecer no print.
QUANTIDADE_FEATURES_ESPERADA = 11
if len(FEATURES_SCORE_POS_VIAGEM) != QUANTIDADE_FEATURES_ESPERADA:
    raise AssertionError(
        f"A allowlist mudou: {len(FEATURES_SCORE_POS_VIAGEM)} features, "
        f"esperadas {QUANTIDADE_FEATURES_ESPERADA}. "
        "Atualize o contrato de entrada da seção 1 antes de seguir."
    )

print("features do score pós-viagem:", len(FEATURES_SCORE_POS_VIAGEM))

### 1.2. Localização e carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a célula falha com a instrução do que fazer, em vez de seguir com uma base vazia.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)

if caminho_base is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Gere-a com a seção 13 de notebooks/pre-processamento.ipynb "
        "ou monte o Drive e ajuste CAMINHOS_POSSIVEIS. Caminhos testados: "
        + ", ".join(str(caminho) for caminho in CAMINHOS_POSSIVEIS)
    )

validar_parquet(caminho_base)
df = pd.read_parquet(caminho_base)

print("base analítica:", caminho_base)
print("linhas:", f"{len(df):,}".replace(",", "."), "| colunas:", df.shape[1])

### 1.3. Diagnóstico da base analítica

A célula confere o que a base precisa ter e interrompe a execução se faltar coluna obrigatória ou feature da allowlist. Falhar aqui é barato; descobrir a ausência depois de treinar não é.

A contagem de colunas `NPS_` e `SUB_` é diagnóstico e não motivo de parada, porque a base analítica carrega essas colunas por construção, como explicado na seção 1. O que não pode acontecer é elas alcançarem a matriz do modelo, e é a seção 1.4 que bloqueia isso.

In [ ]:
COLUNAS_OBRIGATORIAS = {"DETRATOR", "DATA_STD_CONVERTIDA", "ID_GOLDENRECORD"}

faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
if faltando:
    raise KeyError(f"A base não atende ao contrato de entrada. Colunas ausentes: {sorted(faltando)}")

features_ausentes = [coluna for coluna in FEATURES_SCORE_POS_VIAGEM if coluna not in df.columns]
if features_ausentes:
    raise KeyError(f"Features da allowlist ausentes na base: {features_ausentes}")

# Diagnóstico, e não trava: a base analítica carrega as respostas da pesquisa
# por construção, já que DETRATOR é derivado de NPS_PRINCIPAL. O bloqueio
# efetivo acontece sobre a matriz do modelo, na seção 1.4.
colunas_pesquisa = [coluna for coluna in df.columns if coluna.startswith(("NPS_", "SUB_"))]
print("colunas de pesquisa na base analítica:", len(colunas_pesquisa), "(diagnóstico, não bloqueia)")
print("prevalência de Detrator:", f"{df['DETRATOR'].mean():.2%}")

### 1.4. Matriz de modelagem e contrato da matriz

`criar_preprocessador_modelagem` executa em uma chamada a seleção da allowlist, a divisão temporal por Cliente e o ajuste do pré-processador apenas no treino. O retorno traz os dados em três formas: as partições originais, as matrizes transformadas e a lista de colunas numéricas e categóricas.

É aqui que o contrato da matriz é cobrado, nas duas pontas. A verificação de entrada compara o conjunto de features com a allowlist e é a que pega uma allowlist alterada, caso em que a filtragem por construção deixaria de proteger. A de saída confere os nomes gerados pelo pré-processador e pega transformação que reintroduza uma coluna de pesquisa pelo nome. Qualquer uma das duas interrompe a execução.

In [ ]:
(
    preprocessador,
    (x_treino, x_teste, y_treino, y_teste),
    (x_treino_transformado, x_teste_transformado),
    (numericas, categoricas),
    metadados_divisao,
) = criar_preprocessador_modelagem(df)

grupos_treino = df.loc[x_treino.index, "ID_GOLDENRECORD"]

# Contrato da matriz, na entrada do pré-processador.
features_usadas = set(x_treino.columns)
if features_usadas != set(FEATURES_SCORE_POS_VIAGEM):
    raise AssertionError(
        "A matriz não usa exatamente a allowlist do score pós-viagem. "
        f"Sobrando: {sorted(features_usadas - set(FEATURES_SCORE_POS_VIAGEM))}. "
        f"Faltando: {sorted(set(FEATURES_SCORE_POS_VIAGEM) - features_usadas)}."
    )

entrada_proibida = [coluna for coluna in x_treino.columns if coluna.startswith(("NPS_", "SUB_"))]
if entrada_proibida:
    raise AssertionError(f"Coluna de pesquisa entrou na matriz do modelo: {entrada_proibida}")

# Contrato da matriz, na saída do pré-processador.
colunas_matriz = list(preprocessador.get_feature_names_out())
saida_proibida = [coluna for coluna in colunas_matriz if "NPS_" in coluna or "SUB_" in coluna]
if saida_proibida:
    raise AssertionError(f"Coluna de pesquisa apareceu na saída do pré-processador: {saida_proibida}")

print("treino:", x_treino_transformado.shape, "| teste:", x_teste_transformado.shape)
print("numéricas:", len(numericas), "| categóricas:", len(categoricas))
print("prevalência no treino:", f"{y_treino.mean():.2%}", "| no teste:", f"{y_teste.mean():.2%}")
print()
for chave, valor in metadados_divisao.items():
    print(f"{chave}: {valor}")

## 2. Baselines de referência

Um número de desempenho só significa alguma coisa quando existe um piso para compará-lo. Esta seção estabelece dois pisos, e todo modelo apresentado depois é julgado contra eles.

O primeiro é a **regra da classe majoritária**, que não aprende nada e existe para expor o quanto a acurácia engana numa base em que cerca de um em cada cinco Clientes é Detrator. O segundo é uma **regressão logística**, modelo linear e interpretável, que mede quanto do sinal já é capturado sem recorrer a interação entre variáveis. É contra esse segundo piso que a escolha de um método baseado em árvores, na seção 4, precisa se justificar.

Os dois são ajustados apenas na partição de treino e medidos sobre a mesma partição de teste que o modelo candidato usará, sem o que a comparação não seria válida.

**Escopo desta seção.** A acurácia e a precisão média aparecem aqui como piso de comparação, não como a resposta ao item (c) da Seção 4.3. A escolha e a explicação das três métricas oficiais do modelo são daquele item e ficam na seção 6 deste notebook.

### 2.1. Prevalência e função de avaliação

A prevalência da classe Detrator é o número que dá sentido a todo o resto: é ela que a precisão média de um modelo sem informação reproduz, e é dela que a acurácia do baseline trivial se aproxima por complemento.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, average_precision_score

print(f"prevalência de Detrator no treino: {y_treino.mean():.2%}")
print(f"prevalência de Detrator no teste:  {y_teste.mean():.2%}")


def avaliar_no_teste(nome, modelo):
    """Mede um modelo já treinado sobre a partição de teste temporal.

    A acurácia entra ao lado da precisão média de propósito. É a distância
    entre as duas que evidencia o efeito do desbalanceamento, e é isso que
    justifica não apresentar acurácia como medida de qualidade neste problema.
    """
    probabilidades = modelo.predict_proba(x_teste_transformado)[:, 1]
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_teste, modelo.predict(x_teste_transformado)),
        "precisao_media": average_precision_score(y_teste, probabilidades),
    }

### 2.2. Baseline trivial: regra da classe majoritária

O classificador responde sempre a classe mais frequente do treino, que é a de não Detrator. Ele não olha nenhuma feature.

O resultado esperado é uma acurácia próxima de quatro quintos, acompanhada de uma precisão média próxima da prevalência. É essa contradição que interessa: um modelo que erra todos os Detratores da base ainda assim exibe acurácia alta. A probabilidade que ele emite é degenerada, constante para toda linha, e é por isso que a precisão média desaba para o piso.

In [ ]:
trivial = DummyClassifier(strategy="most_frequent", random_state=SEMENTE)
trivial.fit(x_treino_transformado, y_treino)

resultado_trivial = avaliar_no_teste("classe majoritária", trivial)
print(f"acurácia:       {resultado_trivial['acuracia']:.4f}")
print(f"precisão média: {resultado_trivial['precisao_media']:.4f}")

### 2.3. Baseline linear: regressão logística

O segundo piso já aprende, mas apenas efeitos aditivos: cada variável contribui com um peso fixo, sem que o modelo represente combinações entre elas.

O parâmetro de balanceamento de classe é declarado explicitamente. Sem ele a logística se ajusta à classe majoritária e passa a quase não prever Detrator, o que a tornaria indistinguível do baseline trivial e inútil como piso. A matriz chega aqui já escalonada pelo pré-processador da seção 1.4, condição para que os coeficientes sejam comparáveis entre variáveis de unidades diferentes.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression(
    class_weight="balanced",   # declarado: sem isso o modelo colapsa na classe majoritária
    max_iter=5000,          # lbfgs converge em 1010 iterações nesta matriz; 1000 para antes
    random_state=SEMENTE,
)
logistica.fit(x_treino_transformado, y_treino)

resultado_logistica = avaliar_no_teste("regressão logística", logistica)
print(f"acurácia:       {resultado_logistica['acuracia']:.4f}")
print(f"precisão média: {resultado_logistica['precisao_media']:.4f}")

### 2.4. Coeficientes da regressão logística

Os coeficientes ficam acessíveis no notebook para leitura posterior. Eles indicam a direção com que cada variável empurra o risco de detração, e voltam a ser usados na discussão de resultados, quando for preciso dizer se o modelo candidato encontrou algo que o piso linear não via.

In [ ]:
coeficientes_logistica = pd.DataFrame({
    "feature": preprocessador.get_feature_names_out(),
    "coeficiente": logistica.coef_[0],
}).sort_values("coeficiente", key=abs, ascending=False, ignore_index=True)

print("intercepto:", float(logistica.intercept_[0]))
coeficientes_logistica.head(15)

### 2.5. Resultado consolidado dos dois pisos

A tabela abaixo é o piso de comparação que as seções seguintes consomem. A leitura que importa: o baseline trivial exibe a maior acurácia da tabela e a menor precisão média, e a regressão logística inverte essa relação. Qualquer modelo apresentado depois precisa superar a segunda linha na precisão média para que seu ganho seja atribuível ao modelo, e não ao desbalanceamento da base.

In [ ]:
tabela_baselines = pd.DataFrame([resultado_trivial, resultado_logistica])

ganho = resultado_logistica["precisao_media"] - resultado_trivial["precisao_media"]
print(f"ganho da logística sobre o trivial na precisão média: {ganho:+.4f}")
if ganho <= 0:
    print("ATENÇÃO: a logística não superou o piso trivial; revise o balanceamento antes de seguir.")

tabela_baselines

## 3. Validação cruzada por Cliente

`GroupKFold` de 5 folds dentro do treino, agrupando por `ID_GOLDENRECORD` para que nenhum Cliente apareça em dois folds.

*A preencher no card #102.*

## 4. Primeiro modelo candidato

Classificador de gradient boosting sobre árvores, escolha justificada pela ausência de preditor dominante e pela interação entre fidelização e falha operacional documentadas na seção 4.2.1.

*A preencher no card #103.*

## 5. Ajuste de hiperparâmetros

Busca sobre taxa de aprendizado, profundidade e número de iterações, avaliada apenas nos folds de validação.

*A preencher no card #104.*

## 6. Métricas de ordenação do score

Precisão média, área sob a curva ROC e escore de Brier, calculadas sobre o conjunto de teste.

*A preencher no card #105.*

## 7. Limiar operacional e matriz de confusão

Ponto de corte derivado da capacidade diária de contato da equipe de Experiência do Cliente, e não fixado em 0,5 por omissão.

*A preencher no card #106.*

## 8. Curvas ROC e de precisão contra cobertura

Figuras salvas em `assets/`, no mesmo padrão das da seção 4.2.

*A preencher no card #107.*

## 9. Tabela comparativa

Baselines e modelo candidato nas mesmas métricas e sobre a mesma partição de teste.

*A preencher no card #108.*